In [0]:
%run ./lib/00_common

In [0]:
%run ./lib/02_delta_writer

In [0]:
%run ./lib/03_logger_utils

In [0]:
%run ./lib/06_alerting

In [0]:
%run ./lib/07_audit

In [0]:
import pandas as pd

run_id = get_text_widget("run_id", "").strip()
if not run_id:
    raise ValueError("run_id es obligatorio")

alert_email_to = get_text_widget("alert_email_to", "")
gmail_smtp_user = get_text_widget("gmail_smtp_user", "")
gmail_app_password = get_text_widget("gmail_app_password", "")

In [0]:


tables_to_audit = [
    BRONZE_ORDERS_TABLE,
    BRONZE_CUSTOMERS_TABLE,
    SILVER_ORDERS_TABLE,
    SILVER_CUSTOMERS_TABLE,
    GOLD_DAILY_SALES_TABLE,
    GOLD_SALES_BY_COUNTRY_TABLE
]

try:
    stage = "audit_versions"
    dataset = "delta_history"

    log_event(
        spark=spark,
        level="INFO",
        run_id=run_id,
        stage=stage,
        dataset=dataset,
        status="STARTED",
        message="Inicia auditoría de versiones Delta"
    )

    frames = [collect_latest_history_pdf(table_name, run_id) for table_name in tables_to_audit]
    audit_pdf = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

    append_dataset_versions_delta(audit_pdf)

    log_event(
        spark=spark,
        level="INFO",
        run_id=run_id,
        stage=stage,
        dataset=dataset,
        status="OK",
        message="Auditoría de versiones completada",
        rows_ok=len(audit_pdf),
        extra={"tables": tables_to_audit}
    )

except Exception as e:
    notify_failure(
        spark=spark,
        run_id=run_id,
        stage="audit_versions",
        dataset="delta_history",
        error_message=str(e),
        error_code="AUD_001",
        smtp_user=gmail_smtp_user,
        smtp_app_password=gmail_app_password,
        to_email=alert_email_to
    )
    raise

print(f"Audit pandas-first OK. run_id={run_id}")